In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style = 'whitegrid', font_scale = 1.1)
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
df = pd.read_csv('data/raw/dataset.csv')

df.info(), df.describe(), df['track_genre'].value_counts()

In [ ]:
df = df.drop_duplicates(subset=['track_id'], keep = 'first')

audio_features = [
    'danceability', 'energy', 'loudness', 'speechiness', 'acousticness',
    'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'popularity'
]

# оставлю только топ-10 жанров, иначе мультикласс будет слишком размыт
top_genres = df['track_genre'].value_counts().nlargest(10).index
df = df[df['track_genre'].isin(top_genres)].copy()

print(f'Оставлено треков: {len(df)}')
print(f'Жанры для классификации: {top_genres.tolist()}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize = (14, 10))

sns.boxplot(data = df, x = 'track_genre', y = 'energy', hue='track_genre', ax = axes[0, 0], palette='Set2')
axes[0,0].set_title('Energy по жанрам')
axes[0,0].tick_params(axis = 'x', rotation = 45)

sns.boxplot(data = df, x = 'track_genre', y = 'danceability', hue='track_genre' ,ax=axes[0,1], palette = 'Set2')
axes[0,1].set_title('Denceability по жанрам')
axes[0,1].tick_params(axis = 'x', rotation = 45)

sns.scatterplot(data=df, x='energy', y='danceability', hue='track_genre', 
                ax=axes[1,0], alpha=0.6, palette='Set2', legend=False)
axes[1,0].set_title('Energy vs Danceability')

sns.histplot(data = df, x = 'valence', hue = 'track_genre', kde=True,
            ax = axes[1,1], palette = 'Set2', multiple = 'stack')
axes[1,1].set_title('Valence (позитивность трека)')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize = (8,6))
corr = df[audio_features].corr()
sns.heatmap(corr, annot = True, cmap = 'coolwarm', fmt='.2f', linewidths = 0.5)
plt.title('Корреляция между аудио-фичами')
plt.show()

In [ ]:
df_clean = df[audio_features + ['track_genre']].copy()
df_clean = df_clean.dropna()

os.makedirs('data/processed', exist_ok = True)
df_clean.to_csv('data/processed/spotify_clean.csv', index = False)
print('Готово! Очищенный датасет сохранён в data/processed/spotify_clean.csv')
